# Pandas Practice: Bike Share Trip Data

<center><img src="../../assets/nextbike_bike_rental_berlin.jpg" width="800"/></center>

In [ ]:
# --- Starter code: run this cell first ---
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../../data/bike_share_201402_trip_data.csv")
df.head()

## Part 1: Cleaning and Exploring

### Challenge 1: How many trips are in the dataset?

Find the number of observations (rows) in the DataFrame.

In [ ]:
# Number of observations (rows) in the dataset
len(df)

In [ ]:
df['Trip ID'].nunique()

**Explanation**

len(df) returns the row count of the DataFrame, which is the number of trips. df.shape[0] would also work and gives the same number.


### Challenge 2: Make the column names Pythonic

Rename all columns so they are:
- **Lowercase**
- Spaces replaced with `_`
- `#` replaced with `num`

Print the updated column names to verify.

In [ ]:
# Build the new names with chained string replacements, then assign them back.
# Order matters: replace "#" before lowercasing is fine here since "#" has no case.
df.columns = df.columns.str.lower().str.replace(" ", "_").str.replace("#", "num")

# Print the updated names to verify
print(df.columns.tolist())

**Explanation**

We chain vectorised string methods on df.columns to lowercase, replace spaces with underscores, and turn the '#' character into 'num', then assign the result back. This is cleaner than renaming each column by hand.


---

## Part 2: Subscription Analysis

### Challenge 3: Subscription types

How many types of subscription are there? What are they?

In [ ]:
# How many distinct subscription types, and what are they?
print("Number of types:", df["subscription_type"].nunique())
print("Types:", df["subscription_type"].unique())

**Explanation**

nunique() counts the distinct subscription types and unique() lists them. Together they answer both how many and which.


### Challenge 4: Frequency of each subscription type

How many trips were made by each subscription type?

In [ ]:
# value_counts gives the trip count per subscription type
df["subscription_type"].value_counts()

**Explanation**

value_counts() tallies how many rows share each subscription type, giving the trip count per type in one call. It sorts from most to least frequent by default.


### Challenge 5: Visualise subscription frequency, pie chart

Plot the frequency of each subscription type as a pie chart.

In [ ]:
# Reuse the value_counts result and plot it directly as a pie chart
subscription_counts = df["subscription_type"].value_counts()
subscription_counts.plot(kind="pie", autopct="%1.1f%%", ylabel="")
plt.title("Trips by Subscription Type")
plt.show()

**Explanation**

We reuse the value_counts() Series and call .plot(kind='pie') on it. autopct adds percentage labels so the slices are easy to read; pie charts are best when you only have a few categories.


### Challenge 6: Visualise subscription frequency, bar chart

Plot the same data as a bar chart.

In [ ]:
# Same counts, shown as a bar chart for easier comparison of exact values
subscription_counts = df["subscription_type"].value_counts()
subscription_counts.plot(kind="bar")
plt.title("Trips by Subscription Type")
plt.xlabel("Subscription Type")
plt.ylabel("Number of Trips")
plt.show()

**Explanation**

Same counts plotted as a bar chart, which makes it easier to compare exact values than a pie chart. We label the axes so the plot is self-explanatory.


---

## Part 3: Station Analysis

### Challenge 7: Top 10 most popular start stations

Which 10 start stations appear most frequently in the data?

In [ ]:
# value_counts is sorted high-to-low by default, so head(10) gives the busiest
df["start_station"].value_counts().head(10)

**Explanation**

value_counts() is already sorted high-to-low, so head(10) returns the 10 busiest start stations without any extra sorting step.


### Challenge 8: 10 least popular end stations

Which 10 end stations appear least often?

In [ ]:
# tail(10) of a high-to-low count gives the least frequent end stations
df["end_station"].value_counts().tail(10)

**Explanation**

For the least popular end stations we take tail(10) of the high-to-low counts. An alternative is value_counts(ascending=True).head(10), which gives the same stations.


### Challenge 9: Cross-tabulation of start stations by subscription type

Create a table that shows the count of trips for each `start_station` segmented by `subscription_type`, including row and column totals (subtotals).

> Hint: look up `pd.crosstab()` in the documentation.

In [ ]:
# pd.crosstab counts trips per start_station x subscription_type.
# margins=True adds the "All" row and column totals (subtotals).
pd.crosstab(df["start_station"], df["subscription_type"], margins=True)

**Explanation**

pd.crosstab() builds a table of trip counts with start_station as rows and subscription_type as columns. margins=True adds the 'All' row and column with the subtotals the task asks for.


---

## Part 4: Trip Duration

### Challenge 10: Explore duration

What unit do you think the `duration` column uses? Find the shortest and longest trips. How many trips are as short as the minimum?

In [ ]:
# Duration values are large integers, which suggests seconds rather than minutes.
print("Shortest trip:", df["duration"].min())
print("Longest trip:", df["duration"].max())

# How many trips are exactly at the minimum duration?
shortest = df["duration"].min()
print("Trips at the minimum:", (df["duration"] == shortest).sum())

**Explanation**

The large integer values point to seconds as the unit. min() and max() find the shortest and longest trips, and comparing the column to the minimum then summing the booleans counts how many trips share that shortest duration.


### Challenge 11: Define and count "long" trips

Define what you consider a "long" trip (justify your threshold). How many trips meet that definition? What might explain the very long durations?

In [ ]:
# A normal city bike trip is short, so treat anything over 1 hour (3600 seconds)
# as a "long" trip. Comparing the column to the threshold returns a boolean Series,
# and summing it counts the True values.
long_threshold = 3600
print("Long trips (over 1 hour):", (df["duration"] > long_threshold).sum())

# The very long durations are likely bikes that were not docked properly,
# so the rental kept running.
df[df["duration"] > long_threshold]["duration"].describe()

**Explanation**

We pick 1 hour (3600 seconds) as the 'long' threshold since most city bike trips are short, then count trips above it by summing a boolean Series. The extreme durations likely come from bikes that were never docked correctly, so the rental kept counting.


### Challenge 12: Plot the duration distribution

Create a histogram of the `duration` column. Does it give clear insights? If not, filter to a more meaningful range and re-plot.

In [ ]:
# A raw histogram is dominated by a few extreme outliers, so the bars are unreadable.
df["duration"].plot(kind="hist", bins=50)
plt.title("Trip Duration (all trips)")
plt.xlabel("Duration (seconds)")
plt.show()

# Filter to trips under 1 hour to see the meaningful shape of the distribution
short_trips = df[df["duration"] < 3600]
short_trips["duration"].plot(kind="hist", bins=50)
plt.title("Trip Duration (under 1 hour)")
plt.xlabel("Duration (seconds)")
plt.show()

**Explanation**

The first histogram is skewed by a few huge outliers, so the bars are unreadable. Filtering to trips under 1 hour and re-plotting reveals the real shape of typical trip durations.


---

## Part 5: Data Cleaning (Advanced)

### Challenge 13: Normalise station names

The product team wants all station names to be lowercase with underscores as separators.

Example: `South Van Ness at Market` → `south_van_ness_at_market`

Apply this transformation to both the `start_station` and `end_station` columns.

> **Do NOT use a for loop.** Use a vectorised Pandas string method instead, it will be much faster.

In [ ]:
# Vectorised string methods run over the whole column at C speed, no Python loop.
df["start_station"] = df["start_station"].str.lower().str.replace(" ", "_")
df["end_station"] = df["end_station"].str.lower().str.replace(" ", "_")

df[["start_station", "end_station"]].head()

**Explanation**

str.lower() and str.replace(' ', '_') are vectorised, so they transform the whole column at once with no Python for loop, which is both faster and more readable. We do it for both station columns and assign back.


### Challenge 14: Open-ended exploration

Set a 15-minute timer. Use that time to explore the data guided by your own curiosity or hypotheses. What patterns can you find?

> Time-boxing is a useful technique when exploring new data: it prevents you from falling into rabbit holes and forces you to prioritise the most interesting questions.

In [ ]:
# Example exploration: do subscribers and customers take trips of different lengths?
# groupby splits the data by subscription type, then we describe duration for each group.
df.groupby("subscription_type")["duration"].describe()

**Explanation**

An example open-ended question: do the two subscription types differ in trip length? groupby('subscription_type') splits the rows by type and describe() summarises duration for each group so we can compare them.
